<h3>Setup: imports and SCF</h3>

In [ ]:
# Setup: imports and SCF
from pathlib import Path

import qdk_chemistry.plugins.pyscf

from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import available, create, print_settings
from qdk_chemistry.utils import Logger, compute_valence_space_parameters

Logger.set_global_level(Logger.LogLevel.off)

N2_XYZ = Path("../examples/data/stretched_n2.structure.xyz")
structure = Structure.from_xyz_file(N2_XYZ)

# Run SCF with cc-pvdz (same as Chapter 1 output)
scf_solver = create("scf_solver")
E_hf, wfn_hf = scf_solver.run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")
print(f"HF energy: {E_hf:.6f} Hartree")
print("\nCanonical orbital summary:")
print(wfn_hf.get_orbitals().get_summary())

<h3>Compute valence space parameters</h3>

In [ ]:
# Compute valence space parameters
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
print(f"Valence electrons: {num_val_e}")
print(f"Valence orbitals:  {num_val_o}")

# Select the valence subspace using qdk_valence
valence_selector = create(
    "active_space_selector",
    "qdk_valence",
    num_active_electrons=num_val_e,
    num_active_orbitals=num_val_o
)
wfn_valence = valence_selector.run(wfn_hf)

print("\nValence orbital summary:")
print(wfn_valence.get_orbitals().get_summary())

# Get the active space orbital indices for use in localization
valence_indices = wfn_valence.get_orbitals().get_active_space_indices()
print(f"\nActive space indices: {valence_indices}")

<h3>List available orbital localizers</h3>

In [ ]:
# List available orbital localizers
localizers = available("orbital_localizer")
print(f"Available orbital localizers: {localizers}")

<h3>Inspect localizer settings</h3>

In [ ]:
# Inspect localizer settings
for name in localizers:
    print(f"\n--- {name} ---")
    print_settings("orbital_localizer", name)

<h3>Apply MP2 natural orbital localization</h3>

In [ ]:
# Apply MP2 natural orbital localization
localizer_mp2 = create("orbital_localizer", "qdk_mp2_natural_orbitals")

# run() takes the wavefunction and the active space index bounds
wfn_mp2 = localizer_mp2.run(wfn_valence, *valence_indices)

print("Orbital summary after MP2 natural orbital localization:")
print(wfn_mp2.get_orbitals().get_summary())

<h3>Apply Pipek-Mezey localization</h3>

In [ ]:
# Apply Pipek-Mezey localization
localizer_pm = create("orbital_localizer", "qdk_pipek_mezey")
wfn_pm = localizer_pm.run(wfn_valence, *valence_indices)

print("Orbital summary after Pipek-Mezey localization:")
print(wfn_pm.get_orbitals().get_summary())

<h3>Apply VVHV localization</h3>

In [ ]:
# Apply VVHV localization
# VVHV differs from other localizers: it requires ALL virtual orbital indices
# (from n_alpha_electrons to num_molecular_orbitals-1), not just the active space.
localizer_vvhv = create("orbital_localizer", "qdk_vvhv")

num_alpha, num_beta = wfn_valence.get_total_num_electrons()
num_mos = wfn_valence.get_orbitals().get_num_molecular_orbitals()
vvhv_indices = (list(range(num_alpha, num_mos)), list(range(num_beta, num_mos)))

wfn_vvhv = localizer_vvhv.run(wfn_valence, *vvhv_indices)

print("Orbital summary after VVHV localization:")
print(wfn_vvhv.get_orbitals().get_summary())

<h3>Compare all localizer outputs</h3>

In [ ]:
# Compare all localizer outputs
print("=" * 60)
print("CANONICAL (valence)")
print("=" * 60)
print(wfn_valence.get_orbitals().get_summary())

for label, wfn in [("MP2 natural orbitals", wfn_mp2), ("Pipek-Mezey", wfn_pm), ("VVHV", wfn_vvhv)]:
    print("=" * 60)
    print(label)
    print("=" * 60)
    print(wfn.get_orbitals().get_summary())

# What to look for in get_summary():
# - For MP2 natural orbitals: fractional occupations far from 0 or 2 flag strongly correlated orbitals
# - Active space size: a smaller active space with the same chemistry = a cheaper quantum calculation
# - Compare 'Active Orbitals' counts across methods to see how each partitions the orbital space

<h3>Apply Foster-Boys localization via PySCF plugin</h3>

In [ ]:
# Apply Foster-Boys localization via PySCF plugin
# After import it registers 'pyscf_multi' as an additional orbital_localizer
print("Available localizers after PySCF import:", available("orbital_localizer"))

# Inspect the pyscf localizer's configurable methods
print()
print_settings("orbital_localizer", "pyscf_multi")

# Apply Foster-Boys via the PySCF localizer
localizer_fb = create("orbital_localizer", "pyscf_multi", method=???) # fill in this blank
wfn_fb = localizer_fb.run(wfn_valence, *valence_indices)

print("\nOrbital summary after Foster-Boys localization:")
print(wfn_fb.get_orbitals().get_summary())

print("\n--- Built-in Pipek-Mezey (for comparison) ---")
print(wfn_pm.get_orbitals().get_summary())

<h3>Inspect available stability checkers</h3>

In [ ]:
# Inspect available stability checkers
print("Available stability checkers:", available("stability_checker"))
print()
print_settings("stability_checker", "pyscf")
# Key settings to note:
# - internal: tests stability within same wavefunction type (RHF stays RHF)
# - external: tests RHF → UHF instability (broken-symmetry solutions)
# - stability_tolerance: eigenvalue threshold — negative eigenvalue = unstable direction

<h3>Run the stability checker</h3>

In [ ]:
# Run the stability checker
stability_checker = create("stability_checker", "pyscf")
is_stable, stability_result = stability_checker.run(wfn_hf)

print(f"Overall stable:   {is_stable}")
print(f"Internal stable:  {stability_result.is_internal_stable()}")
print(f"External stable:  {stability_result.is_external_stable()}")
print(f"\nInternal eigenvalues: {stability_result.get_internal_eigenvalues()}")
print(f"External eigenvalues: {stability_result.get_external_eigenvalues()}")

# Interpreting the result:
# - Internal instability: a lower-energy solution exists within the same spin symmetry
# - External instability (RHF → UHF): a broken-symmetry unrestricted solution exists
#   This is common for stretched bonds — N₂ at 1.27 Å is expected to be externally unstable
#   because restricted HF cannot describe bond breaking correctly

# What to do when unstable:
# An external instability confirms that single-reference HF is an inadequate description.
# The correct response is NOT to re-run SCF, but to move to a multi-reference treatment —
# exactly what the active space + CASCI workflow in Chapters 3-4 provides.
# The localized MP2 natural orbital wavefunction (wfn_mp2) is the right starting point.
if not is_stable:
    print("\n→ Wavefunction is unstable. Proceed with multi-reference active space workflow (Chapter 3).")
else:
    print("\n→ Wavefunction is stable. Single-reference treatment may be adequate.")